# CHSH-style EstimatorV2 sweep

Evaluate two Bell-pair correlations across a measurement-basis sweep with Qiskit and MettleQ estimators.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

EstimatorV2 evaluates observables without materializing counts. Sweeping the measurement basis reveals the Bell-pair correlation curve.

In [2]:
angles = np.linspace(0.0, np.pi, 9)
observables = [SparsePauliOp("ZZ"), SparsePauliOp("ZX")]
circuits = []
for angle in angles:
    circuit = QuantumCircuit(2)
    circuit.h(0)
    circuit.cx(0, 1)
    circuit.ry(float(angle), 0)
    circuits.append(circuit)
pubs = [(circuit, observables) for circuit in circuits]

def run_reference():
    return np.asarray([StatevectorEstimator().run([pub]).result()[0].data.evs for pub in pubs])

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(run_reference)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
mettleq_pubs = [(transpile(circuit, backend, optimization_level=1), observables) for circuit in circuits]
estimator = MettleQEstimatorV2(backend=backend)

def run_mettleq():
    return np.asarray([estimator.run([pub]).result()[0].data.evs for pub in mettleq_pubs])

candidate, mettleq_ms, _ = benchmark(run_mettleq)
error = max_abs_error(reference, candidate)
method, device = qiskit_selection(estimator)

## 4. Check correctness before discussing speed

Every expectation value in the two-observable sweep is compared within an absolute tolerance.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/04_estimator_chsh.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="EstimatorV2 correlation curve atol=2e-6",
    passed=error <= 2e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_expectation_error": error, "angles": angles, "reference": reference, "mettleq": candidate},
)


Comparison summary
------------------
Correctness contract: PASS — EstimatorV2 correlation curve atol=2e-6
SDK reference median: 6.451 ms
MettleQ median:       29.114 ms
Timing interpretation: the SDK reference was 4.513x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "EstimatorV2 correlation curve atol=2e-6", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"angles": [0.0, 0.39269908169872414, 0.7853981633974483, 1.1780972450961724, 1.5707963267948966, 1.9634954084936207, 2.356194490192345, 2.748893571891069, 3.141592653589793], "max_expectation_error": 1.1920928932873665e-07, "mettleq": [[0.9999999403953552, 0.0], [0.9238794445991516, 0.3826834261417389], [0.7071067094802856, 0.7071067690849304], [0.3826833665370941, 0.9238795042037964], [0.0, 0.9999998807907104], [-0.3826833963394165, 0.

## What should you conclude?

Estimator workloads become attractive when many large circuits share a stable execution path; this small sweep emphasizes semantics.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.